# NTFS Timestomping Detection Tool v2.0
# Step 2: Feature Engineering

---

## Overview

This notebook extracts 51+ forensic features from the merged data:
- **Timestamp Analysis**: Deltas, direction changes, zero nanoseconds
- **File Characteristics**: Extensions, attributes, path depth
- **Event Patterns**: Frequency, temporal clustering
- **Cross-Artifact Validation**: LogFile + UsnJrnl consistency scores

These features enable the ML model to detect timestamp manipulation patterns.

---

## Input

`merged_forensic_data.csv` from Step 1

## Output

`forensic_features.csv` - Ready for ML detection

---

## 1. Setup

In [84]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from datetime import timedelta
import re

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


## 2. Load Merged Data

In [85]:
# Input/Output paths
INPUT_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output')
OUTPUT_DIR = INPUT_DIR  # Same directory

input_file = INPUT_DIR / 'merged_forensic_data.csv'

print("=" * 80)
print("LOADING MERGED DATA")
print("=" * 80)
print(f"\nInput: {input_file}")

if not input_file.exists():
    raise FileNotFoundError(f"Input file not found: {input_file}\nPlease run 01_Load_Data.ipynb first!")

df = pd.read_csv(input_file, low_memory=False)

print(f"\n✓ Loaded {len(df):,} records")
print(f"✓ {len(df.columns)} columns")
print(f"\nSource distribution:")
print(df['source'].value_counts())

LOADING MERGED DATA

Input: /Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/merged_forensic_data.csv

✓ Loaded 5,289 records
✓ 25 columns

Source distribution:
source
usnjrnl_only    5139
both             149
logfile_only       1
Name: count, dtype: int64


## 3. Feature Engineering Functions

These functions extract forensic features used by the trained model.

In [86]:
def parse_lf_detail_field(df):
    """
    Parse LogFile lf_detail field to extract timestamp changes.
    
    Example: 'ModifiedTime : 2023-12-31 01:15:23 -> 2000-01-01 08:00:00'
    Extracts: before/after timestamps, zero nanoseconds flag
    """
    print("\n  Parsing lf_detail field...")
    
    # Initialize columns
    for ts_type in ['creation', 'modified', 'accessed', 'mft_modified']:
        df[f'lf_{ts_type}_time_before'] = None
        df[f'lf_{ts_type}_time_after'] = None
    df['zero_in_nanoseconds'] = False
    
    # Initialize copied_from_file as object type (for one-hot encoding compatibility)
    df['copied_from_file'] = None  # Will be 'True' or remain None (becomes 'missing')
    
    # Regex pattern for timestamp manipulation
    pattern = r'(\w+Time)\s*:\s*([\d\-:\s]+)\s*->\s*([\d\-:\s]+)'
    
    parsed_count = 0
    for idx, row in df[df['lf_detail'].notna()].iterrows():
        detail = str(row['lf_detail'])
        
        # Extract timestamp changes
        match = re.search(pattern, detail)
        if match:
            ts_type = match.group(1).replace('Time', '').lower()
            if ts_type == 'mftmodified':
                ts_type = 'mft_modified'
            
            df.at[idx, f'lf_{ts_type}_time_before'] = match.group(2).strip()
            df.at[idx, f'lf_{ts_type}_time_after'] = match.group(3).strip()
            parsed_count += 1
        
        # Check for zero nanoseconds
        if 'Zero in 100-nanoseconds' in detail or 'zero in 100-nanoseconds' in detail:
            df.at[idx, 'zero_in_nanoseconds'] = True
        
        # Check for copied timestamps (set as string 'True' for one-hot encoding)
        if 'Copied from File' in detail or 'same as' in detail.lower():
            df.at[idx, 'copied_from_file'] = 'True'
    
    print(f"    ✓ Parsed {parsed_count:,} lf_detail entries")
    print(f"    ✓ Zero nanoseconds: {df['zero_in_nanoseconds'].sum():,} events")
    print(f"    ✓ Copied timestamps: {(df['copied_from_file'] == 'True').sum():,} events")
    
    return df

In [87]:
def extract_timestamp_delta_features(df):
    """
    Calculate timestamp change magnitudes and directions.
    """
    print("\n  Extracting timestamp delta features...")
    
    for ts_type in ['creation', 'modified', 'accessed', 'mft_modified']:
        before_col = f'lf_{ts_type}_time_before'
        after_col = f'lf_{ts_type}_time_after'
        delta_col = f'{ts_type}_time_delta_days'
        direction_col = f'{ts_type}_time_changed_to_past'
        
        # Parse timestamps
        before_dt = pd.to_datetime(df[before_col], errors='coerce')
        after_dt = pd.to_datetime(df[after_col], errors='coerce')
        
        # Calculate delta in days
        delta = (before_dt - after_dt).dt.total_seconds() / 86400
        df[delta_col] = delta
        
        # Direction: True if timestamp moved to past, None if no data
        # Convert to object type with 'True'/'missing' for one-hot encoding compatibility
        df[direction_col] = None
        df.loc[delta.notna() & (delta > 0), direction_col] = 'True'
        df.loc[delta.notna() & (delta <= 0), direction_col] = 'False'
        # Leave as None (will become 'missing' during one-hot encoding) where no data
    
    # Count non-null deltas
    delta_cols = [f'{t}_time_delta_days' for t in ['creation', 'modified', 'accessed', 'mft_modified']]
    total_deltas = df[delta_cols].notna().sum().sum()
    print(f"    ✓ Calculated {total_deltas:,} timestamp deltas")
    
    return df

In [88]:
def extract_file_characteristics(df):
    """
    Extract file-based features: extension, attributes, path depth.
    """
    print("\n  Extracting file characteristics...")
    
    # File extension suspicious check
    suspicious_extensions = ['.exe', '.dll', '.sys', '.bat', '.cmd', '.ps1', '.vbs', '.js']
    df['has_suspicious_extension'] = df['filename'].fillna('').str.lower().apply(
        lambda x: any(x.endswith(ext) for ext in suspicious_extensions)
    )
    
    # Filename length
    df['filename_length'] = df['filename'].fillna('').str.len()
    
    # File attributes from UsnJrnl
    df['is_executable'] = df['usn_file_attribute'].fillna('').str.contains('Executable', case=False, na=False)
    df['is_system_file'] = df['usn_file_attribute'].fillna('').str.contains('System', case=False, na=False)
    df['is_hidden_file'] = df['usn_file_attribute'].fillna('').str.contains('Hidden', case=False, na=False)
    df['is_archive'] = df['usn_file_attribute'].fillna('').str.contains('Archive', case=False, na=False)
    
    # Path depth (number of subdirectories)
    df['path_depth'] = df['filepath'].fillna('').str.count(r'[\\|/]')
    
    print(f"    ✓ Suspicious extensions: {df['has_suspicious_extension'].sum():,} files")
    print(f"    ✓ Executable files: {df['is_executable'].sum():,}")
    print(f"    ✓ Average path depth: {df['path_depth'].mean():.1f}")
    
    return df

In [89]:
def extract_temporal_features(df):
    """
    Extract event frequency and temporal clustering features.
    """
    print("\n  Extracting temporal features...")
    
    # Sort by time for windowing
    df = df.sort_values('eventtime').reset_index(drop=True)
    
    # Event frequency per file
    df['event_frequency_per_file'] = df.groupby('merge_key')['merge_key'].transform('count')
    
    # Events in time windows (1 min, 5 min)
    df['events_in_1min_window'] = 0
    df['events_in_5min_window'] = 0
    
    for idx, row in df.iterrows():
        if pd.notna(row['eventtime']):
            # 1-minute window
            window_1min = df[
                (df['eventtime'] >= row['eventtime'] - timedelta(minutes=1)) &
                (df['eventtime'] <= row['eventtime'] + timedelta(minutes=1))
            ]
            df.at[idx, 'events_in_1min_window'] = len(window_1min)
            
            # 5-minute window
            window_5min = df[
                (df['eventtime'] >= row['eventtime'] - timedelta(minutes=5)) &
                (df['eventtime'] <= row['eventtime'] + timedelta(minutes=5))
            ]
            df.at[idx, 'events_in_5min_window'] = len(window_5min)
    
    # Time since previous/until next event
    df['time_since_previous_event_seconds'] = df['eventtime'].diff().dt.total_seconds()
    df['time_until_next_event_seconds'] = df['eventtime'].diff(-1).abs().dt.total_seconds()
    
    print(f"    ✓ Max events in 1-min window: {df['events_in_1min_window'].max():.0f}")
    print(f"    ✓ Max events in 5-min window: {df['events_in_5min_window'].max():.0f}")
    
    return df

In [90]:
def extract_cross_artifact_features(df):
    """
    Extract cross-artifact validation scores.
    """
    print("\n  Extracting cross-artifact features...")
    
    # Source confidence score (0-3)
    df['source_confidence_score'] = 0
    df.loc[df['has_logfile_evidence'], 'source_confidence_score'] += 1
    df.loc[df['has_usnjrnl_evidence'], 'source_confidence_score'] += 1
    df.loc[df['source'] == 'both', 'source_confidence_score'] += 1
    
    # UsnJrnl pattern flags
    df['usn_basic_info_change'] = df['usn_event_info'].fillna('').str.contains('Basic_Info_Change', case=False, na=False)
    df['usn_file_closed'] = df['usn_event_info'].fillna('').str.contains('File_Closed', case=False, na=False)
    
    # Complete manipulation pattern (BASIC_INFO_CHANGE + CLOSE)
    df['usn_complete_manipulation_pattern'] = df['usn_basic_info_change'] & df['usn_file_closed']
    
    # Cross-artifact validation score (0-3)
    df['cross_artifact_validation_score'] = 0
    df.loc[df['source'] == 'both', 'cross_artifact_validation_score'] += 2  # Both sources
    df.loc[df['lf_event'].fillna('').str.contains('Time Reversal', case=False), 'cross_artifact_validation_score'] += 1  # Time Reversal
    df.loc[df['usn_complete_manipulation_pattern'], 'cross_artifact_validation_score'] += 1  # Complete USN pattern
    
    # Timestamp manipulation pattern score (0-3)
    df['timestamp_manipulation_pattern_score'] = 0
    df.loc[df['zero_in_nanoseconds'], 'timestamp_manipulation_pattern_score'] += 1
    # Check if copied_from_file == 'True' (it's now an object type)
    df.loc[df['copied_from_file'] == 'True', 'timestamp_manipulation_pattern_score'] += 1
    # Check if any timestamp changed to past (now object type with 'True' values)
    changed_to_past_mask = (
        (df['creation_time_changed_to_past'] == 'True') |
        (df['modified_time_changed_to_past'] == 'True') |
        (df['accessed_time_changed_to_past'] == 'True')
    )
    df.loc[changed_to_past_mask, 'timestamp_manipulation_pattern_score'] += 1
    
    print(f"    ✓ Average source confidence: {df['source_confidence_score'].mean():.2f}")
    print(f"    ✓ Average cross-artifact score: {df['cross_artifact_validation_score'].mean():.2f}")
    print(f"    ✓ Average manipulation pattern score: {df['timestamp_manipulation_pattern_score'].mean():.2f}")
    
    return df

In [91]:
def extract_additional_features(df):
    """
    Extract remaining features for model compatibility.
    """
    print("\n  Extracting additional features...")
    
    # File system tunneling confidence (0 or 1 based on tunneling detection from Phase 1)
    df['file_system_tunneling_confidence'] = 0
    if 'is_tunneling' in df.columns:
        df.loc[df['is_tunneling'], 'file_system_tunneling_confidence'] = 1
    
    # Attribute change indicator
    df['has_attribute_change'] = df['lf_event'].fillna('').str.contains('Changing FileAttribute', case=False, na=False)
    
    # Copied timestamp indicator (convert from object 'True' to boolean True)
    df['has_timestamp_copied_from_file'] = (df['copied_from_file'] == 'True')
    
    # Zero nanoseconds from LogFile
    df['zero_nanoseconds_logfile'] = df['zero_in_nanoseconds']
    
    # Event vs modified time delta
    df['event_vs_modified_after_days'] = None
    if 'lf_modified_time_after' in df.columns:
        modified_after = pd.to_datetime(df['lf_modified_time_after'], errors='coerce')
        event_time = pd.to_datetime(df['eventtime'], errors='coerce')
        df['event_vs_modified_after_days'] = (event_time - modified_after).dt.total_seconds() / 86400
    
    # Event frequency per case (set to same as per file for single case)
    df['event_frequency_per_case'] = df['event_frequency_per_file']
    
    print(f"    ✓ Tunneling events: {df['file_system_tunneling_confidence'].sum():.0f}")
    print(f"    ✓ Attribute changes: {df['has_attribute_change'].sum():,}")
    
    return df

## 4. Apply Feature Engineering

In [92]:
print("=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

# Convert eventtime to datetime if not already
df['eventtime'] = pd.to_datetime(df['eventtime'], errors='coerce')

# Apply all feature extraction functions
print("\n1. Parsing LogFile detail field...")
df = parse_lf_detail_field(df)

print("\n2. Extracting timestamp deltas...")
df = extract_timestamp_delta_features(df)

print("\n3. Extracting file characteristics...")
df = extract_file_characteristics(df)

print("\n4. Extracting temporal features...")
print("   (This may take a few minutes for large datasets...)")
df = extract_temporal_features(df)

print("\n5. Extracting cross-artifact features...")
df = extract_cross_artifact_features(df)

print("\n6. Extracting additional features...")
df = extract_additional_features(df)

print("\n" + "=" * 80)
print("✓ Feature engineering complete")
print("=" * 80)
print(f"\nTotal features: {len(df.columns)} columns")

FEATURE ENGINEERING

1. Parsing LogFile detail field...

  Parsing lf_detail field...
    ✓ Parsed 150 lf_detail entries
    ✓ Zero nanoseconds: 0 events
    ✓ Copied timestamps: 1 events

2. Extracting timestamp deltas...

  Extracting timestamp delta features...
    ✓ Calculated 150 timestamp deltas

3. Extracting file characteristics...

  Extracting file characteristics...
    ✓ Suspicious extensions: 471 files
    ✓ Executable files: 0
    ✓ Average path depth: 4.6

4. Extracting temporal features...
   (This may take a few minutes for large datasets...)

  Extracting temporal features...


    ✓ Max events in 1-min window: 482
    ✓ Max events in 5-min window: 776

5. Extracting cross-artifact features...

  Extracting cross-artifact features...
    ✓ Average source confidence: 1.06
    ✓ Average cross-artifact score: 0.51
    ✓ Average manipulation pattern score: 0.03

6. Extracting additional features...

  Extracting additional features...
    ✓ Tunneling events: 130
    ✓ Attribute changes: 0

✓ Feature engineering complete

Total features: 67 columns


## 5. Data Cleanup and Column Management

Remove unnecessary columns and add required columns for model compatibility.

In [93]:
print("=" * 80)
print("DATA CLEANUP")
print("=" * 80)

# Add missing columns for model compatibility
print("\n1. Adding missing columns...")

# eventtime_dt (datetime version of eventtime)
df['eventtime_dt'] = df['eventtime']
print("   ✓ Added eventtime_dt")

# Remove unnecessary columns
print("\n2. Removing unnecessary columns...")

columns_to_remove = [
    'lf_detail',  # Already parsed
    'lf_creation_time',  # Empty
    'lf_modified_time',  # Empty
    'lf_accessed_time',  # Empty
    'lf_mft_modified_time',  # Empty
    'usn_source_info',  # Not used by model
    'usn_carving_flag'  # Not used by model
]

removed = []
for col in columns_to_remove:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)
        removed.append(col)

print(f"   ✓ Removed {len(removed)} columns:")
for col in removed:
    print(f"     - {col}")

print("\n" + "=" * 80)
print(f"✓ Cleanup complete: {len(df.columns)} columns remaining")
print("=" * 80)

DATA CLEANUP

1. Adding missing columns...
   ✓ Added eventtime_dt

2. Removing unnecessary columns...
   ✓ Removed 7 columns:
     - lf_detail
     - lf_creation_time
     - lf_modified_time
     - lf_accessed_time
     - lf_mft_modified_time
     - usn_source_info
     - usn_carving_flag

✓ Cleanup complete: 61 columns remaining


## 6. Save Features

In [94]:
print("=" * 80)
print("SAVING FEATURES")
print("=" * 80)

output_file = OUTPUT_DIR / 'forensic_features.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n✓ Saved features to:")
print(f"   {output_file}")
print(f"\n   Total records: {len(df):,}")
print(f"   Total columns: {len(df.columns)}")
print(f"\n✓ Ready for detection (Step 3)")

SAVING FEATURES

✓ Saved features to:
   /Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/forensic_features.csv

   Total records: 5,289
   Total columns: 61

✓ Ready for detection (Step 3)


## 7. Feature Summary

In [95]:
print("=" * 80)
print("FEATURE SUMMARY")
print("=" * 80)

# List all engineered features
base_cols = ['lf_lsn', 'usn_usn', 'eventtime', 'filename', 'filepath', 'lf_event', 'lf_detail', 
             'usn_event_info', 'usn_source_info', 'usn_file_attribute', 'merge_key', 'source']
feature_cols = [col for col in df.columns if col not in base_cols]

print(f"\nEngineered Features ({len(feature_cols)} total):\n")
for i, col in enumerate(sorted(feature_cols), 1):
    non_null = df[col].notna().sum()
    pct = non_null / len(df) * 100
    print(f"{i:2}. {col:45} {non_null:6,} / {len(df):,} ({pct:5.1f}%)")

print("\n" + "=" * 80)
print("✓ Feature engineering complete!")
print("=" * 80)

FEATURE SUMMARY

Engineered Features (51 total):

 1. accessed_time_changed_to_past                      0 / 5,289 (  0.0%)
 2. accessed_time_delta_days                           0 / 5,289 (  0.0%)
 3. copied_from_file                                   1 / 5,289 (  0.0%)
 4. creation_time_changed_to_past                      1 / 5,289 (  0.0%)
 5. creation_time_delta_days                           1 / 5,289 (  0.0%)
 6. cross_artifact_validation_score                5,289 / 5,289 (100.0%)
 7. event_frequency_per_case                       5,289 / 5,289 (100.0%)
 8. event_frequency_per_file                       5,289 / 5,289 (100.0%)
 9. event_vs_modified_after_days                     148 / 5,289 (  2.8%)
10. events_in_1min_window                          5,289 / 5,289 (100.0%)
11. events_in_5min_window                          5,289 / 5,289 (100.0%)
12. eventtime_dt                                   5,288 / 5,289 (100.0%)
13. file_system_tunneling_confidence               5,289 / 5,2

---

## ✓ Step 2 Complete!

**Next:** Run `03_Run_Detection.ipynb` to apply the trained model and detect timestomped files.

---